<a href="https://colab.research.google.com/github/icosahedron31/Facial-Expression-Recognition/blob/main/CNN_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle wandb onnx -Uq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 23.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! mkdir ~/.kaggle

In [ ]:
! mkdir -p ~/.kaggle && echo KGAT_8cb0c83c63cdaf3612ac74923b3aa519 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:01<00:00, 218MB/s]



In [ ]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tdola23 (tdola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Imports**

In [ ]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


**Read split dataset**

In [ ]:
df = pd.read_csv('../content/train.csv')
train_ds, test_ds = train_test_split(df, test_size = 0.3, random_state=42)
test_ds, val_ds = train_test_split(test_ds, test_size = 0.5, random_state=42)
train_ds.head(), val_ds.head()

(       emotion                                             pixels
 14421        0  157 154 157 137 94 99 115 104 120 110 128 98 1...
 9226         4  11 13 17 20 20 21 24 29 28 28 30 33 40 50 57 6...
 994          2  242 250 154 76 80 69 57 53 50 59 55 45 46 49 5...
 28675        0  111 111 112 110 111 111 109 106 99 88 44 68 12...
 4600         3  7 1 4 17 9 11 25 30 33 31 23 38 29 42 56 48 60...,
        emotion                                             pixels
 17147        3  136 136 137 136 138 137 137 135 133 102 52 36 ...
 14040        3  45 26 33 65 47 36 37 35 36 66 88 69 53 38 47 3...
 21306        3  80 78 78 77 76 76 80 115 132 111 93 80 86 92 8...
 22365        3  250 250 250 251 251 251 252 159 58 69 50 44 42...
 22196        1  255 255 255 255 253 255 165 85 62 57 63 72 90 ...)

**Transform and dataset**

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
class CustomImageDataset(Dataset):
  def __init__(self, df, transform=None) :
    self.df = df
    self.transform = transform
  def __getitem__(self, index) :
    rows = self.df.iloc[index]
    pixels = rows["pixels"]
    label = torch.tensor(rows["emotion"], dtype=torch.long).to(device)
    pixels = np.fromstring(pixels, dtype=np.uint8, sep=" ").reshape(48, 48)
    pixels = self.transform(pixels).to(device)
    return pixels, label
  def __len__(self) :
    return len(self.df)

In [ ]:
train_ds = CustomImageDataset(train_ds, transform)
val_ds = CustomImageDataset(val_ds, transform)

**Models**

cnn + transformer

In [ ]:
import torch
import torch.nn as nn

class CNNTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.batchnorm1 = nn.BatchNorm2d(16)
        self.batchnorm2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.layer = nn.TransformerEncoderLayer(
            d_model = 128,
            dropout=0.3,
            nhead = 1,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.linear = nn.Linear(128, 7)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.batchnorm1(x)
        x = self.relu(self.conv2(x))
        x = self.batchnorm2(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer_encoder(x)
        x = x.mean(dim = 1)
        x = self.linear(x)
        return x

Bigger feature extraction and increased transformer layers


In [ ]:
import torch
import torch.nn as nn

class DeepCNNTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.batchnorm1 = nn.BatchNorm2d(16)
        self.batchnorm2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1)
        self.batchnorm4 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv5 = nn.Conv2d(128, 256, kernel_size = 3, padding = 1)
        self.batchnorm5 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.layer = nn.TransformerEncoderLayer(
            d_model = 256,
            dropout=0.3,
            nhead = 1,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.linear = nn.Linear(256, 7)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.batchnorm1(x)
        x = self.relu(self.conv2(x))
        x = self.batchnorm2(x)
        x = self.relu(self.conv3(x))
        x = self.batchnorm3(x)
        x = self.relu(self.conv4(x))
        x = self.batchnorm4(x)
        x = self.relu(self.conv5(x))
        x = self.batchnorm5(x)
        x = self.pool(x)

        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer_encoder(x)
        x = x.mean(dim = 1)
        x = self.linear(x)
        return x

In [ ]:
import torch
import torch.nn as nn

class DeepCNNTransformerDropout(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.batchnorm1 = nn.BatchNorm2d(16)
        self.batchnorm2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1)
        self.batchnorm4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128, 256, kernel_size = 3, padding = 1)
        self.batchnorm5 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout2D = nn.Dropout2d(0.3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = 256,
            dropout=0.3,
            nhead = 1,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.linear = nn.Linear(256, 7)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.batchnorm1(x)


        x = self.relu(self.conv2(x))
        x = self.batchnorm2(x)


        x = self.relu(self.conv3(x))
        x = self.batchnorm3(x)
        x = self.dropout2D(x)

        x = self.relu(self.conv4(x))
        x = self.batchnorm4(x)
        x = self.dropout2D(x)

        x = self.relu(self.conv5(x))
        x = self.batchnorm5(x)
        x = self.dropout2D(x)
        x = self.pool(x)

        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer_encoder(x)
        x = x.mean(dim = 1)
        x = self.linear(x)
        return x

In [ ]:
import torch
import torch.nn as nn

class PositionalEmbedder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.batchnorm1 = nn.BatchNorm2d(16)
        self.batchnorm2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1)
        self.batchnorm4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128, 256, kernel_size = 3, padding = 1)
        self.batchnorm5 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout2D = nn.Dropout2d(0.3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = 256,
            dropout=0.3,
            nhead = 4,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.linear = nn.Linear(256, 7)
        self.pos = nn.Parameter(torch.randn(1, 400, 256))
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.batchnorm1(x)


        x = self.relu(self.conv2(x))
        x = self.batchnorm2(x)


        x = self.relu(self.conv3(x))
        x = self.batchnorm3(x)
        x = self.dropout2D(x)

        x = self.relu(self.conv4(x))
        x = self.batchnorm4(x)
        x = self.dropout2D(x)

        x = self.relu(self.conv5(x))
        x = self.batchnorm5(x)
        x = self.dropout2D(x)
        x = self.pool(x)

        x = x.flatten(2).permute(0, 2, 1)
        x = self.pos + x
        x = self.transformer_encoder(x)
        x = x.mean(dim = 1)
        x = self.linear(x)
        return x

**Train Loop**

In [ ]:
import copy
def train_loop(train_ds, val_ds, model_name, learning_rate, batch_size, model, EPOCHS=30,
               weight_decay=None) :

  train_dataloader = DataLoader(train_ds, batch_size, shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size, shuffle=False)
  criterion = nn.CrossEntropyLoss()
  best_model = model
  best_acc_val = 0
  if weight_decay :
    optimizer = Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  else :
    optimizer = Adam(model.parameters(), lr = learning_rate)
  for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0
    model.train()
    for inputs, labels in train_dataloader:
      optimizer.zero_grad()
      outputs = model(inputs).to(device)
      train_loss = criterion(outputs, labels)
      total_loss_train += train_loss.item()
      train_loss.backward()

      train_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
      total_acc_train += train_acc
      optimizer.step()
    model.eval()
    with torch.no_grad():
      for inputs, labels in val_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        val_loss = criterion(outputs, labels)
        total_loss_val += val_loss.item()

        val_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
        total_acc_val += val_acc
    if total_acc_val > best_acc_val :
        best_acc_val = total_acc_val
        best_model = copy.deepcopy(model.state_dict())
        if epoch > 10 :
          torch.save(best_model, f"{model_name}.pt")

    print(f'''Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/train_ds.__len__(), 4)} Train Accuracy {round((total_acc_train)/train_ds.__len__() * 100, 4)}
                Validation Loss: {round(total_loss_val/val_ds.__len__(), 4)} Validation Accuracy: {round((total_acc_val)/val_ds.__len__() * 100, 4)}''')


    wandb.log({
        "epoch": epoch,
        "train_loss": total_loss_train / train_ds.__len__(),
        "train_acc": round((total_acc_train)/train_ds.__len__() * 100, 4),
        "val_loss": round(total_loss_val/val_ds.__len__(), 4),
        "val_acc": round((total_acc_val)/val_ds.__len__() * 100, 4)
    })

  return best_model

**Training**

**Overfitting small dataset with small model**

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = CNNTransformer().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="overfitting_small_ds_simple_numlayer3",
    config = {
    "project": "fer",
    "model": "SimpleTransformer",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


/tmp/ipykernel_1874/858025401.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/100, Train Loss: 0.2495 Train Accuracy 18.0
                Validation Loss: 0.234 Validation Accuracy: 25.0
Epoch 2/100, Train Loss: 0.2288 Train Accuracy 28.0
                Validation Loss: 0.2262 Validation Accuracy: 28.0
Epoch 3/100, Train Loss: 0.2267 Train Accuracy 28.0
                Validation Loss: 0.2188 Validation Accuracy: 30.0
Epoch 4/100, Train Loss: 0.2277 Train Accuracy 31.0
                Validation Loss: 0.2197 Validation Accuracy: 29.0
Epoch 5/100, Train Loss: 0.2206 Train Accuracy 29.0
                Validation Loss: 0.2157 Validation Accuracy: 32.0
Epoch 6/100, Train Loss: 0.2189 Train Accuracy 36.0
                Validation Loss: 0.2142 Validation Accuracy: 31.0


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Subset

model = CNNTransformer().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleTransformer",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds, val_ds, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "small_transformer.pt")
wandb.finish()


Overfitting small dataset with larger model

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = DeepCNNTransformer().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="overfitting_small_ds_simple_numlayer3_deepcnn",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleTransformerDeep",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


In [ ]:
from torch.utils.data import Subset

model = DeepCNNTransformer().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "BiggerCNNTransformer",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds, val_ds, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "bigger_transformer.pt")
wandb.finish()


Dropout + Deep CNN

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = DeepCNNTransformerDropout().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="overfitting_small_ds_dropout_less_aggressive_deepcnn",
    config = {
    "project": "ml_hw_04",
    "model": "DeepCNN + dropout",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


/tmp/ipykernel_1874/3133527388.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/100, Train Loss: 0.2514 Train Accuracy 19.0
                Validation Loss: 0.3353 Validation Accuracy: 15.0
Epoch 2/100, Train Loss: 0.2254 Train Accuracy 24.0
                Validation Loss: 0.2994 Validation Accuracy: 14.0
Epoch 3/100, Train Loss: 0.2246 Train Accuracy 31.0
                Validation Loss: 0.2551 Validation Accuracy: 22.0
Epoch 4/100, Train Loss: 0.2307 Train Accuracy 24.0
                Validation Loss: 0.2186 Validation Accuracy: 32.0
Epoch 5/100, Train Loss: 0.2273 Train Accuracy 22.0
                Validation Loss: 0.208 Validation Accuracy: 35.0
Epoch 6/100, Train Loss: 0.2217 Train Accuracy 29.0
                Validation Loss: 0.2066 Validation Accuracy: 38.0
Epoch 7/100, Train Loss: 0.2214 Train Accuracy 25.0
                Validation Loss: 0.2103 Validation Accuracy: 33.0
Epoch 8/100, Train Loss: 0.2169 Train Accuracy 29.0
                Validation Loss: 0.1989 Validation Accuracy: 39.0
Epoch 9/100, Train Loss: 0.2099 Train Accuracy 30.0
     

epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
train_acc,▁▁▂▂▃▂▃▄▃▄▄▄▄▅▆▆▅▆▆▇▆▆▇▇▇▇▇▇▇█▇▇▇█▇████▇
train_loss,█▇▇▇▇▆▆▆▅▆▅▅▅▄▄▄▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁
val_acc,▁▂▂▃▃▃▃▃▄▅▆▆▇▇██████████████████████████
val_loss,█▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,99
train_acc,92
train_loss,0.02575
val_acc,99
val_loss,0.0048


In [ ]:
from torch.utils.data import Subset

model = DeepCNNTransformerDropout().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="dropout_deep_cnn",
    config = {
    "project": "ml_hw_04",
    "model": "BigCNNTransformerWithDropout",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds, val_ds, "cnnTransformer", 0.0001, 32, model, 30)
torch.save(model, "dropout_cnn_transformer.pt")
wandb.finish()


/tmp/ipykernel_1874/3133527388.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/30, Train Loss: 0.0556 Train Accuracy 25.8609
                Validation Loss: 0.056 Validation Accuracy: 28.3956
Epoch 2/30, Train Loss: 0.0529 Train Accuracy 31.4988
                Validation Loss: 0.0508 Validation Accuracy: 36.104
Epoch 3/30, Train Loss: 0.0502 Train Accuracy 35.6787
                Validation Loss: 0.0493 Validation Accuracy: 39.8653
Epoch 4/30, Train Loss: 0.0485 Train Accuracy 38.336
                Validation Loss: 0.0474 Validation Accuracy: 41.1888
Epoch 5/30, Train Loss: 0.0472 Train Accuracy 40.7444
                Validation Loss: 0.048 Validation Accuracy: 42.3264
Epoch 6/30, Train Loss: 0.0463 Train Accuracy 41.7546
                Validation Loss: 0.0464 Validation Accuracy: 44.1375
Epoch 7/30, Train Loss: 0.0453 Train Accuracy 43.755
                Validation Loss: 0.0452 Validation Accuracy: 46.111
Epoch 8/30, Train Loss: 0.0447 Train Accuracy 44.2227
                Validation Loss: 0.0445 Validation Accuracy: 47.0165
Epoch 9/30, Train Loss

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███████
train_loss,█▇▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇████▇▇
val_loss,█▆▅▄▄▃▃▃▃▃▃▃▂▂▃▂▂▂▁▂▂▂▁▁▁▂▁▁▁▂
epoch,29
train_acc,55.5235
train_loss,0.03626
val_acc,51.5672
val_loss,0.0434


positional embedding

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = PositionalEmbedder().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="overfitting_small_ds_pos_embedding",
    config = {
    "project": "ml_hw_04",
    "model": "DeepCNN + dropout",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(small_ds, small_ds, "cnnTransformer", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


Epoch 1/100, Train Loss: 0.2518 Train Accuracy 17.0
                Validation Loss: 0.2402 Validation Accuracy: 14.0
Epoch 2/100, Train Loss: 0.2318 Train Accuracy 26.0
                Validation Loss: 0.2539 Validation Accuracy: 15.0
Epoch 3/100, Train Loss: 0.2274 Train Accuracy 25.0
                Validation Loss: 0.231 Validation Accuracy: 26.0
Epoch 4/100, Train Loss: 0.2284 Train Accuracy 24.0
                Validation Loss: 0.2173 Validation Accuracy: 31.0
Epoch 5/100, Train Loss: 0.2209 Train Accuracy 32.0
                Validation Loss: 0.212 Validation Accuracy: 35.0
Epoch 6/100, Train Loss: 0.222 Train Accuracy 22.0
                Validation Loss: 0.2072 Validation Accuracy: 36.0
Epoch 7/100, Train Loss: 0.2116 Train Accuracy 35.0
                Validation Loss: 0.2048 Validation Accuracy: 36.0
Epoch 8/100, Train Loss: 0.2107 Train Accuracy 34.0
                Validation Loss: 0.2023 Validation Accuracy: 36.0
Epoch 9/100, Train Loss: 0.2092 Train Accuracy 33.0
       

epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇█████
train_acc,▁▂▃▂▃▃▄▄▄▆▅▅▇▆▇▇▇▆▇▇▇▇███▇████▇███████▇█
train_loss,█▇▇▇▇▇▆▆▅▅▄▄▄▄▄▄▃▂▂▃▂▂▂▂▂▂▁▁▂▁▂▁▂▁▁▁▁▁▁▂
val_acc,▁▁▂▃▃▃▃▄▄▅▆▆▆▆▆▆▇▇█▇▇█▇█████████████████
val_loss,██▇▇▇▆▆▆▅▄▄▃▃▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,99
train_acc,99
train_loss,0.00625
val_acc,100
val_loss,0.0004


In [ ]:
from torch.utils.data import Subset

model = DeepCNNTransformerDropout().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="pos_embedding",
    config = {
    "project": "ml_hw_04",
    "model": "pos_embedding",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds, val_ds, "pos_embedding", 0.0001, 32, model, 30)
torch.save(model, "pos_embedding.pt")
wandb.finish()


/tmp/ipykernel_1874/3133527388.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/30, Train Loss: 0.0557 Train Accuracy 25.9853
                Validation Loss: 0.0553 Validation Accuracy: 28.2099
Epoch 2/30, Train Loss: 0.0536 Train Accuracy 30.2448
                Validation Loss: 0.0523 Validation Accuracy: 33.968
Epoch 3/30, Train Loss: 0.0515 Train Accuracy 34.2705
                Validation Loss: 0.051 Validation Accuracy: 37.172
Epoch 4/30, Train Loss: 0.0492 Train Accuracy 37.51
                Validation Loss: 0.049 Validation Accuracy: 41.746
Epoch 5/30, Train Loss: 0.0477 Train Accuracy 39.8487
                Validation Loss: 0.0493 Validation Accuracy: 40.3297
Epoch 6/30, Train Loss: 0.0464 Train Accuracy 41.9188
                Validation Loss: 0.0462 Validation Accuracy: 44.2768
Epoch 7/30, Train Loss: 0.0451 Train Accuracy 43.7152
                Validation Loss: 0.0447 Validation Accuracy: 45.8324
Epoch 8/30, Train Loss: 0.0442 Train Accuracy 44.8497
                Validation Loss: 0.045 Validation Accuracy: 46.8772
Epoch 9/30, Train Loss: 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█▇██████
train_loss,█▇▇▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▃▅▄▅▆▆▆▆▇▇▇▇▇▇▇█▇▇▇█████████
val_loss,█▇▆▅▅▄▃▃▂▄▃▃▂▂▂▂▃▂▂▂▂▁▁▁▁▁▂▁▂▂
epoch,29
train_acc,57.091
train_loss,0.03514
val_acc,54.9338
val_loss,0.0418


**Data Augmentation**

In [ ]:
from PIL import Image
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05)
    ),
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
train_ds_aug, test_ds_aug = train_test_split(df, test_size = 0.3, random_state=42)
test_ds_aug, val_ds_aug = train_test_split(test_ds, test_size = 0.5, random_state=42)

train_ds_augment = CustomImageDataset(train_ds_aug, transform)
val_ds_augment = CustomImageDataset(val_ds_aug, val_transform)

In [ ]:
from torch.utils.data import Subset

model = DeepCNNTransformer().to(device)

wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="deep_numlayer3_augment",
    config = {
    "project": "ml_hw_04",
    "model": "BiggerCNNTransformer",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds_augment, val_ds_augment, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "bigger_transformer.pt")
wandb.finish()


/tmp/ipykernel_1874/1423932178.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/30, Train Loss: 0.0541 Train Accuracy 29.3541
                Validation Loss: 0.0514 Validation Accuracy: 35.9034
Epoch 2/30, Train Loss: 0.0483 Train Accuracy 38.6445
                Validation Loss: 0.0473 Validation Accuracy: 41.5699
Epoch 3/30, Train Loss: 0.0456 Train Accuracy 43.3569
                Validation Loss: 0.0449 Validation Accuracy: 45.3785
Epoch 4/30, Train Loss: 0.0436 Train Accuracy 46.5167
                Validation Loss: 0.0435 Validation Accuracy: 46.9113
Epoch 5/30, Train Loss: 0.0423 Train Accuracy 47.9797
                Validation Loss: 0.0421 Validation Accuracy: 48.3976
Epoch 6/30, Train Loss: 0.0409 Train Accuracy 50.1443
                Validation Loss: 0.042 Validation Accuracy: 50.0697
Epoch 7/30, Train Loss: 0.0401 Train Accuracy 51.1146
                Validation Loss: 0.0428 Validation Accuracy: 49.1407
Epoch 8/30, Train Loss: 0.0391 Train Accuracy 51.8163
                Validation Loss: 0.0417 Validation Accuracy: 50.627
Epoch 9/30, Train 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
train_loss,█▆▆▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▄▅▅▆▅▆▆▇▇▆▇▇▇▆▇▇▇▇█▇▇███▇███
val_loss,█▆▅▄▄▄▄▃▃▃▂▂▂▂▂▃▂▂▂▂▁▂▂▁▁▁▂▁▂▂
epoch,29
train_acc,63.8137
train_loss,0.02984
val_acc,56.8974
val_loss,0.0382


In [ ]:
model = DeepCNNTransformer().to(device)
model.load_state_dict(torch.load("cnn_transformer.pt"))

/tmp/ipykernel_1874/1423932178.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


<All keys matched successfully>

In [ ]:
wandb.init(
    project="ml_hw_04",
    group="cnnTransformer",
    name="deep_numlayer3_augment_continued",
    config = {
    "project": "ml_hw_04",
    "model": "BiggerCNNTransformer",

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "cnnTransformer"
        }
    }
)
model = train_loop(train_ds_augment, val_ds_augment, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "bigger_transformer.pt")
wandb.finish()

Epoch 1/30, Train Loss: 0.0302 Train Accuracy 63.4206
                Validation Loss: 0.0383 Validation Accuracy: 56.758
Epoch 2/30, Train Loss: 0.0299 Train Accuracy 63.8286
                Validation Loss: 0.0378 Validation Accuracy: 56.7116
Epoch 3/30, Train Loss: 0.0296 Train Accuracy 64.1869
                Validation Loss: 0.0377 Validation Accuracy: 56.5258
Epoch 4/30, Train Loss: 0.0291 Train Accuracy 64.5601
                Validation Loss: 0.0381 Validation Accuracy: 56.8509
Epoch 5/30, Train Loss: 0.0289 Train Accuracy 65.2319
                Validation Loss: 0.038 Validation Accuracy: 56.4329
Epoch 6/30, Train Loss: 0.0286 Train Accuracy 65.3115
                Validation Loss: 0.0378 Validation Accuracy: 57.8263
Epoch 7/30, Train Loss: 0.0285 Train Accuracy 65.6947
                Validation Loss: 0.0391 Validation Accuracy: 56.6187
